In [15]:
import os
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
os.getcwd()

'C:\\Users\\MADHULIKA'

In [6]:
# Install BeautifulSoup4 for reading and extracting data from HTML files
!pip install beautifulsoup4 pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Step 1: Create sample HTML pages
html_pages = {
    "A.html": '<html><body><h1>Page A</h1><a href="B.html">B</a> <a href="C.html">C</a></body></html>',
    "B.html": '<html><body><h1>Page B</h1><a href="C.html">C</a></body></html>',
    "C.html": '<html><body><h1>Page C</h1><a href="A.html">A</a></body></html>',
    "D.html": '<html><body><h1>Page D</h1><a href="C.html">C</a></body></html>'
}
# Loop through each HTML file name and its corresponding HTML content
for filename, content in html_pages.items():

    # Open the file in write mode ("w")
    # If the file does not exist, it will be created
    with open(filename, "w") as f:

        # Write the HTML content into the file
        f.write(content)

# Display the names of all HTML files that were created
print("Sample HTML pages created: ", list(html_pages.keys()))

Sample HTML pages created:  ['A.html', 'B.html', 'C.html', 'D.html']


In [12]:
# Step 2: Extract links using BeautifulSoup
pages = list(html_pages.keys())

links = {}
for page in pages:

    with open(page, "r") as f:
        soup = BeautifulSoup(f, "html.parser")
        hrefs = [a['href'] for a in soup.find_all('a', href=True)]
        links[page] = hrefs


# Print a heading
print("\n Web Graph (Extracted from HTML):\n")


# Display each page and the pages it links to
for p, outlinks in links.items():
    print(f"{p} → {outlinks}")


 Web Graph (Extracted from HTML):

A.html → ['B.html', 'C.html']
B.html → ['C.html']
C.html → ['A.html']
D.html → ['C.html']


In [9]:
# Step 3: Build the transition matrix
N = len(pages)
M = np.zeros((N, N))
for i, p in enumerate(pages):

    if links[p]:
        for q in links[p]:
            j = pages.index(q)
            M[j][i] = 1 / len(links[p])

    else:
        M[:, i] = 1 / N


# Display the transition matrix
print("\nTransition Matrix (M):\n", M)


Transition Matrix (M):
 [[0.  0.  1.  0. ]
 [0.5 0.  0.  0. ]
 [0.5 1.  0.  1. ]
 [0.  0.  0.  0. ]]


In [13]:
# Step 4: PageRank calculation
def pagerank(M, d=0.85, iterations=100):
    N = M.shape[0]
    PR = np.ones(N) / N

    for _ in range(iterations):
        PR = (1 - d) / N + d * M.dot(PR)
    return PR
ranks = pagerank(M)

df = pd.DataFrame({
    "Page": pages,
    "PageRank Score": ranks
})

print("\nFinal PageRank Scores:\n")
print(df.sort_values("PageRank Score", ascending=False))


Final PageRank Scores:

     Page  PageRank Score
2  C.html        0.394149
0  A.html        0.372527
1  B.html        0.195824
3  D.html        0.037500


In [14]:
# Save to CSV
# Save the PageRank results DataFrame into a CSV file
df.to_csv("pagerank_results.csv", index=False)

# Display a confirmation message
print("\nSaved results to 'pagerank_results.csv'")


Saved results to 'pagerank_results.csv'
